<b>Alcoa Example: demand and bids for one year</b>

| SKU | Demand (MT) |
|-----|-------------|
| 1   |  403,000    |
| 2   |  24,000     |
| 3   |  800        |
| 4   |  85,000     |
| 5   |  41,000     |
| 6   |  133,000    |
| 7   |  894,000    |
| 8   |  123,000    |

*Price bid per MT*

SKU| V1       | V2       | V3       |
---|----------|----------|----------|
1|  7.922   |  9.125   |  9.200   |
2|  9.493   |  -       |  5.732   |
3|  15.250  |  9.930   |  11.720  |
4|  8.257   |  9.542   |  9.240   |
5|  12.471  |  -       |  -       |
6|  21.383  |  13.977  |  11.050  |
7|  7.413   |  -       |  5.575   |
8|  10.644  |  9.660   |  10.220  |

*Rules*
- If all 3 suppliers bid the SKU, no one supplier can have more than 67% of the volume for that SKU
- If only 2 suppliers bid the SKU, no one supplier can have more than 80% of the volume for that SKU
- No supplier can receive more than 50% of overall global volume

In [1]:
# data
Demand = { 1: 403000, 2: 24000, 3: 800, 4: 85000, 5: 41000, 6: 133000, 7: 894000, 8: 123000 }
Bid = { (1,'V1'): 7.922, (1,'V2'): 9.125, (1,'V3'): 9.2, (2,'V1'): 9.493, (2,'V3'): 5.732, (3,'V1'): 15.25, (3,'V2'): 9.93, (3,'V3'): 11.72, (4,'V1'): 8.257, (4,'V2'): 9.542, (4,'V3'): 9.24, (5,'V1'): 12.471, (6,'V1'): 21.383, (6,'V2'): 13.977, (6,'V3'): 11.05, (7,'V1'): 7.413, (7,'V3'): 5.575, (8,'V1'): 10.644, (8,'V2'): 9.66, (8,'V3'): 10.22 }

In [2]:
from docplex.mp.model import Model
mdl = Model()

In [3]:
# define the set of vendors (needed for later for-loops)
Vendors = {'V1', 'V2', 'V3'}

In [4]:
# variables
# note: we can inherit the dictionary from Bid directly (from its keys)
purchased = mdl.continuous_var_dict(Bid, lb=0, name="purchase")

In [5]:
# objective: minimize total costs
mdl.minimize(mdl.sum(Bid[i,j]*purchased[i,j] for (i,j) in Bid))

In [6]:
# constraints: meet demand
for i in Demand:
    mdl.add_constraint(mdl.sum(purchased[i,j] for j in Vendors if (i,j) in purchased)==Demand[i])

In [7]:
# constraints: if all 3 suppliers bid the SKU, no one supplier can have more than 67% of the volume
#              else if only 2 suppliers bid the SKU, no one supplier can have more than 80% of the volume
for i in Demand: # iterate over all SKUs
    # determine how many vendors bid for SKU i
    VendorsForSKU = {j for j in Vendors if (i,j) in Bid}
    # distinguish the two cases: either three or two vendors for the SKU
    if len(VendorsForSKU) == 3:
        for j in VendorsForSKU:
            mdl.add_constraint(purchased[i,j] <= 0.67*Demand[i])
    elif len(VendorsForSKU) == 2:
        for j in VendorsForSKU:
            mdl.add_constraint(purchased[i,j] <= 0.8*Demand[i])

In [8]:
# constraints: no supplier can receive more than 50% of overall global volume
TotalVolume = sum(Demand[i] for i in Demand)
for j in Vendors:
    mdl.add_constraint(mdl.sum(purchased[i,j] for i in Demand if (i,j) in purchased) <= 0.5*TotalVolume)

In [9]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0,status='optimal')

In [10]:
mdl.print_solution()

objective: 12892786.146
  purchase_1_V1=270010.000
  purchase_1_V2=132990.000
  purchase_2_V1=4800.000
  purchase_2_V3=19200.000
  purchase_3_V2=536.000
  purchase_3_V3=264.000
  purchase_4_V1=56950.000
  purchase_4_V2=28050.000
  purchase_5_V1=41000.000
  purchase_6_V2=43890.000
  purchase_6_V3=89110.000
  purchase_7_V1=178800.000
  purchase_7_V3=715200.000
  purchase_8_V1=12464.000
  purchase_8_V2=82410.000
  purchase_8_V3=28126.000
